In [2]:
from transformers import AutoTokenizer
from google.colab import drive

drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/NLP/resources/"

tokenizer_en = AutoTokenizer.from_pretrained(base_path + "tokenizer_en")
tokenizer_fr = AutoTokenizer.from_pretrained(base_path + "tokenizer_fr")

V_encoder = tokenizer_fr.vocab_size
V_decoder = tokenizer_en.vocab_size

fr_pad_id = tokenizer_fr.pad_token_id
en_pad_id = tokenizer_en.pad_token_id
bos_id = tokenizer_en.bos_token_id
eos_id = tokenizer_en.eos_token_id

print(f"Encoder Vocab Size V(e): {V_encoder}")
print(f"Decoder Vocab Size V(d): {V_decoder}")
print(f"Padding Token ID: {fr_pad_id}")
print(f"BOS Token ID: {bos_id}")
print(f"EOS Token ID: {eos_id}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Encoder Vocab Size V(e): 3200
Decoder Vocab Size V(d): 3200
Padding Token ID: 3
BOS Token ID: 1
EOS Token ID: 2


In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk

train_hf = load_from_disk(base_path + "parallel_en_fr_corpus/train")
valid_hf = load_from_disk(base_path + "parallel_en_fr_corpus/validation")
test_hf = load_from_disk(base_path + "parallel_en_fr_corpus/test")

class NMTDataset(Dataset):
    def __init__(self, dataset_hf, tokenizer_fr, tokenizer_en, max_len=32):
        self.data = dataset_hf
        self.tok_fr = tokenizer_fr
        self.tok_en = tokenizer_en
        self.max_len = max_len

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Extract the text for the current row
        fr_text = self.data[idx]['text_fr'] 
        en_text = self.data[idx]['text_en']

        # Tokenize French (Encoder Input)
        enc_encoding = self.tok_fr(
            fr_text, 
            max_length=self.max_len, 
            padding='max_length', 
            truncation=True, 
            return_tensors="pt"
        )

        # Tokenize English (Decoder Input)
        dec_encoding = self.tok_en(
            en_text, 
            max_length=self.max_len, 
            padding='max_length', 
            truncation=True, 
            return_tensors="pt"
        )
        
        # Remove the batch dimension added by PyTorch tensors and returns IDs
        return {
            "encoder_input": enc_encoding["input_ids"].squeeze(0),
            "decoder_input": dec_encoding["input_ids"].squeeze(0)
        }
    
# Create PyTorch datasets
train_dataset = NMTDataset(train_hf, tokenizer_fr, tokenizer_en, max_len=32)
valid_dataset = NMTDataset(valid_hf, tokenizer_fr, tokenizer_en, max_len=32)
test_dataset = NMTDataset(test_hf, tokenizer_fr, tokenizer_en, max_len=32)

# Initialize DataLoaders with a batch size of 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [11]:
class TransformerEmbeddings(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=32):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)

    def forward(self, input_ids):
        # input_ids: (batch_size, seq_length)
        seq_length = input_ids.size(1)

        # Create position IDs for each token in the sequence
        position_ids = torch.arange(seq_length, device=input_ids.device).unsqueeze(0)
        
        token_embeds = self.token_embedding(input_ids)      # Shape: [batch_size, seq_len, d_model]
        position_embeds = self.position_embedding(position_ids)     # Shape: [1, seq_len, d_model]
        
        return token_embeds + position_embeds

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=32, num_heads=4):
        super().__init__()
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_H = d_model // num_heads
        
        # Linear layers for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Final linear layer after concatenation
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        pass